In [1]:
import pandas as pd

In [2]:
fake = pd.read_csv("Fake.csv")
true = pd.read_csv("True.csv")

In [3]:
fake.head()

,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017"
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017"
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017"
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017"


In [4]:
true.head()

,title,text,subject,date
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017"
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017"
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017"
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017"


In [5]:
fake["label"] = 0
true["label"] = 1

In [6]:
true.head()

,title,text,subject,date,label
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017",1
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017",1
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017",1
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017",1
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017",1


In [7]:
news = pd.concat([fake, true], ignore_index=True)

In [8]:
news = news.sample(frac=1, random_state=42).reset_index(drop=True)

In [9]:
news.head()

,title,text,subject,date,label
0,Ben Stein Calls Out 9th Circuit Court: Committ...,"21st Century Wire says Ben Stein, reputable pr...",US_News,"February 13, 2017",0
1,Trump drops Steve Bannon from National Securit...,WASHINGTON (Reuters) - U.S. President Donald T...,politicsNews,"April 5, 2017",1
2,Puerto Rico expects U.S. to lift Jones Act shi...,(Reuters) - Puerto Rico Governor Ricardo Rosse...,politicsNews,"September 27, 2017",1
3,OOPS: Trump Just Accidentally Confirmed He Le...,"On Monday, Donald Trump once again embarrassed...",News,"May 22, 2017",0
4,Donald Trump heads for Scotland to reopen a go...,"GLASGOW, Scotland (Reuters) - Most U.S. presid...",politicsNews,"June 24, 2016",1


In [10]:
news.shape

(44898, 5)

In [11]:
#Data cleaning and preprocessing
import re
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\navee\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [12]:
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()

In [13]:
import re
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

# 3000 random rows
messages_sample = news.sample(n=3000, random_state=42).reset_index(drop=True)

ps = PorterStemmer()
stop_words = set(stopwords.words('english'))

corpus = []

for i in range(len(messages_sample)):
    review = re.sub('[^a-zA-Z]', ' ', messages_sample['text'][i])
    review = review.lower()
    review = review.split()

    review = [ps.stem(word) for word in review if word not in stop_words]
    review = ' '.join(review)

    corpus.append(review)

In [14]:
corpus

['well take long short time sinc american kinda sorta elect donald trump pussygrabb chief trump appoint bona fide white nationalist high level posit perform numer action month ago consid extrem unlik nightmar scenario latest seem muslim registri media told us trump seriou might actual happen trump regim muslim immigr come unit state trump pal call terror prone countri soon regist check regularli govern trump advis kansa secretari state kri kobach way kobach say trump administr aim quickli wast taxpay dollar wall move forward propos get thing roll registri immigr muslim countri reuter report kansa secretari state kri kobach help write tough immigr law arizona elsewher said interview trump polici advis also discuss draft propos consider reinstat registri immigr muslim countri kobach media report say key member trump transit team said particip regular confer call dozen trump immigr advis past two three month trump transit team respond request confirm kobach role presid elect commit follow

In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000)

X = tfidf.fit_transform(corpus).toarray()

In [16]:
X

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [17]:
y = messages_sample['label'].values

In [18]:
y

array([0, 1, 1, ..., 0, 0, 1])

In [19]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42
)

In [20]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()

model.fit(X_train, y_train)

LogisticRegression()

In [21]:
y_pred = model.predict(X_test)

In [22]:
from sklearn.metrics import accuracy_score

accuracy_score(y_test, y_pred)

0.955

In [23]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.95      0.96      0.96       310
           1       0.96      0.95      0.95       290

    accuracy                           0.95       600
   macro avg       0.96      0.95      0.95       600
weighted avg       0.96      0.95      0.95       600



In [24]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)
print(cm)

[[297  13]
 [ 14 276]]


In [25]:
from sklearn.metrics import roc_auc_score

y_prob = model.predict_proba(X_test)[:,1]

auc = roc_auc_score(y_test, y_prob)

print(auc)

0.9931701890989989


In [26]:
from sklearn.model_selection import GridSearchCV

params = {
    "C":[0.01,0.1,1,10],
    "solver":["liblinear","lbfgs"]
}

grid = GridSearchCV(
    LogisticRegression(),
    params,
    cv=5
)

grid.fit(X_train,y_train)

print(grid.best_params_)

{'C': 10, 'solver': 'liblinear'}


In [27]:
import pickle

pickle.dump(model,open("model.pkl","wb"))

pickle.dump(tfidf,open("vectorizer.pkl","wb"))